# Harmonise (step 8a): cohort, clock, units and cleaning rules

Turns the concept map into long tables, one parquet per site and concept group, in `derived/harmonised/` (row-level: stays in the SDE, git-ignored). Columns: site, stay_key, concept, hour (hours since the stay's t0, true elapsed time), value, value_text (text concepts), flag, source.

Rules (see `DATA_ISSUES.md` and the `harmonise.py` docstring): stays over 6 hours only; all chart times to UTC before subtracting t0 (SMH chart times are UTC, the rest London local); haemoglobin to g/L; haematocrit to percent; placeholder zeros to missing; GOSH diastolic equal to systolic to missing; GOSH base excess derived from pH and bicarbonate and flagged `derived`; tidal volume unit stripped; I:E ratio as the x of 1:x.

Fluids and drugs are not in this notebook (see `harmonise_fluids`, next step). Only counts and hours leave the SDE.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import pandas as pd
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import concepts as C
import harmonise as H

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
assembled_dir = open_picu_dir / "derived" / "assembled"
reports_dir = open_picu_dir / "reports"
harmonised_dir = open_picu_dir / "derived" / "harmonised"
static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
today = date.today().isoformat()
co = H.cohort(static)
print(H.__file__)
print(co.groupby("site").size().to_dict(), "stays over 6 hours of", static.groupby("site").size().to_dict())

## 1. Is the clock right?
Median hours of the first heart rate after t0, by site and season. If the time zones are handled correctly, summer and winter should agree (an error would show as a one-hour step). The share before t0 or more than 24 h after shows how far chart times sit from the recorded admission.

In [ ]:
clock = H.time_alignment_check(assembled_dir, static)
clock

## 2. Harmonise every non-fluid, non-drug concept
Reads each source with its row times, applies the rules and writes one parquet per site and group. The report has counts only: rows read, outside the cohort, with a bad time, non-numeric, set missing, converted, derived, and the 5th, 50th and 95th percentile of `hour`. This is slow (millions of rows); run it once.

In [ ]:
report = H.harmonise_all(assembled_dir, static, harmonised_dir)
report.to_csv(Path(reports_dir) / "harmonise_report.csv", index=False)
report.drop(columns="cohort_stays")

## 2b. Fluids
Fluids have their own definitions (see the `harmonise.py` section header and `DATA_ISSUES.md` G8, S3, R6): hourly ml, with `intake_total` at all sites and `intake_iv` (infusion, colloids and drug volume; no blood, enteral or TPN) at RBH and SMH only. GOSH hourly amounts come from differencing the 24 h running totals before the Epic switch and from `caboodle_fluid_balance` after it. Nothing is removed: hours above the 99.9th percentile of ml per kg per hour (per site and concept) are flagged `outlier`, and the cap values are reported so the choice of percentile can be reviewed.

In [ ]:
fl_report, fl_caps = H.harmonise_fluids(assembled_dir, static, harmonised_dir)
fl_report.to_csv(Path(reports_dir) / "harmonise_fluids_report.csv", index=False)
fl_caps.to_csv(Path(reports_dir) / "harmonise_fluids_caps.csv", index=False)
display(fl_report[["site", "concept", "source", "stays", "rows", "rows_gap", "rows_outlier", "hour_median"]])
display(fl_caps)

## 3. Save the results for review

In [ ]:
out = Path(reports_dir) / f"harmonise_review_{today}.md"
out.write_text("## Clock check\n\n```\n" + clock.to_string() + "\n```\n\n"
               "## Report\n\n```\n" + report.drop(columns="cohort_stays").to_string() + "\n```\n\n"
               "## Fluids report\n\n```\n" + fl_report.to_string() + "\n```\n\n"
               "## Fluids caps (ml per kg per hour)\n\n```\n" + fl_caps.to_string() + "\n```\n")
print("written:", out)